![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Processing blood gases

This notebook imports and processes blood gas data and exports it into a pickle archive.

- Clinical data for cases where ventilation recordings (>1 minutes) are also available: **819 cases**
- One or more blood gas is available in **759 cases**

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

### 1. Import the required libraries and setting options

In [4]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import re
import pickle

from scipy import stats
from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('mode.chained_assignment', None) 

In [5]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("SciPy version: {}".format(sp.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
SciPy version: 1.16.2
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [7]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Directory on external drive to read the clinical data and blood gases from
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'ventilator_data', 'Fabian', 'fabian_patient_data_all')

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian', 'Analyses', 'analysis_1_1100')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on external drive to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/ventilator_data/Fabian/fabian_patient_data_all',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian/Analyses/analysis_1_1100',
 '/Volumes/GB_1/data_dump/fabian')

### 3. Import clinical data from pickle archive

In [9]:
with open(os.path.join(DATA_DUMP, 'clin_df_1_1100.pickle'), 'rb') as handle:
    clin_df = pickle.load(handle)
len(clin_df)

819

### 4. Import all clinical data containing blood gases

In [11]:
# import text files in a dictionary
clin_dict = {}
for fname in os.listdir(DIR_READ):
    if not fname.startswith('.'): # disregard hidden files
        fhandle = open(os.path.join('%s' % DIR_READ, fname), 'r', encoding = 'cp1252')
        clin_dict[fname[:-4]] = fhandle.read() # use the filenames without the .txt extension as keys
        fhandle.close()

In [12]:
# Only keep blood gases which also have ventilator data
clin_dict = {key: value for key, value in clin_dict.items() if key in clin_df.index}
len(clin_dict)

819

In [13]:
gas_dict = {}
# Remove clinical details preceding the blood gases
for key, value in clin_dict.items():
    try:
        gas_dict[key] = value[value.index('Astrup'):]
    except ValueError:
        print(key, 'has no blood gas')
len(gas_dict)

819

In [14]:
gas_dict_2 = {}

for key, value in gas_dict.items():
    gas_dict_2[key] = {}
    
    for i, gas in enumerate(value.split('Astrup')[1:]):
        gas_dict_2[key][i] = {}
        items = gas.split('\n')[1:-1]
        for item in items:
            name, value = item.split(':')
            if value.strip() == '':
                break
            else:
                gas_dict_2[key][i][name.strip()] = value.strip()

In [15]:
for case in gas_dict_2:
    for gas in sorted(gas_dict_2[case].keys()):
        if gas_dict_2[case][gas] == {}:
            del gas_dict_2[case][gas]

In [16]:
gas_frames = {}
for case in gas_dict_2.keys():
    gas_frames[case] = DataFrame(gas_dict_2[case])

In [17]:
def time_changer(rec):
    a = clin_df.loc[rec]['Recording start'].date()
    for column in gas_frames[rec]:
        b = gas_frames[rec].loc['Time', column]
        c = datetime.strptime(str(b), '%H%M').time()
        # This str() is needed here because AL000665 (and only that) is interpreted as Datetime
        d = datetime.combine(a, c)
        gas_frames[rec].loc['Time', column] = d  

In [18]:
for case in gas_frames: 
    # AL000310 has an abnormal time stamp for first gas ("70:19")
    if case == 'AL000310':
        continue
    #print(case)
    time_changer(case)

In [19]:
for case in sorted(gas_frames.keys()):
    try:
        gas_frames[case] =  gas_frames[case].T
        gas_frames[case]['Time'] = gas_frames[case]['Time'].astype('datetime64[ns]')
        gas_frames[case] = gas_frames[case].set_index('Time')
        
    except Exception as e:
        print('No blood gas for %s' % case)
        del gas_frames[case]

No blood gas for AL000042
No blood gas for AL000051
No blood gas for AL000070
No blood gas for AL000133
No blood gas for AL000144
No blood gas for AL000169
No blood gas for AL000172
No blood gas for AL000290
No blood gas for AL000294
No blood gas for AL000308
No blood gas for AL000333
No blood gas for AL000353
No blood gas for AL000360
No blood gas for AL000369
No blood gas for AL000428
No blood gas for AL000436
No blood gas for AL000440
No blood gas for AL000464
No blood gas for AL000471
No blood gas for AL000475
No blood gas for AL000492
No blood gas for AL000504
No blood gas for AL000518
No blood gas for AL000537
No blood gas for AL000548
No blood gas for AL000554
No blood gas for AL000557
No blood gas for AL000606
No blood gas for AL000627
No blood gas for AL000653
No blood gas for AL000667
No blood gas for AL000668
No blood gas for AL000683
No blood gas for AL000703
No blood gas for AL000748
No blood gas for AL000758
No blood gas for AL000777
No blood gas for AL000783
No blood gas

In [20]:
len(gas_frames)

759

### 5. Quality control of blood gases

#### Combine all gases to a single DataFrame

In [23]:
gas_frames_all = pd.concat(gas_frames)

In [24]:
gas_frames_all = gas_frames_all[['pH', 'pCO2', 'pO2', 'HCO3', 'ABE', 'FiO2', 'Type']]
for column in ['pH', 'pCO2', 'pO2', 'HCO3', 'ABE', 'FiO2']:
    gas_frames_all[column] = gas_frames_all[column].dropna().astype('float', errors='ignore')
for column in ['Type']:
    gas_frames_all[column] = gas_frames_all[column].astype('category')

In [25]:
gas_frames_all[gas_frames_all['pO2'] == '']

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,


In [26]:
gas_frames_all.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1382 entries, ('AL000003', Timestamp('2017-03-24 18:20:00')) to ('AL001086', Timestamp('2020-09-21 11:30:00'))
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   pH      1382 non-null   float64 
 1   pCO2    1378 non-null   float64 
 2   pO2     1370 non-null   float64 
 3   HCO3    1340 non-null   float64 
 4   ABE     1332 non-null   float64 
 5   FiO2    1119 non-null   float64 
 6   Type    1071 non-null   category
dtypes: category(1), float64(6)
memory usage: 136.7+ KB


#### Manually review outlier with outlier values in gases and correct as appropriate

- pH <6.6 or pH >7.5
- pCO2 <20 mmHg or >130 mmHg
- ABE <-30 or >20 

Only correct trivial ones. 

- For example, "7" is sometimes mis-recognised as "2" by OCR. 
- Other times, the decimal point is clearly in the wrong place.

Only correct those ones where the other values in the blood gas are consistent with the change you are making. Otherwise, remove the clearly impossible values.

In [28]:
# These are all correct
gas_frames_all[gas_frames_all['pH'] < 6.6].sort_values('pH', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AL000528,2019-05-09 22:53:00,6.42,99.0,51.8,3.4,-34.0,0.7,Capillaris
AL000007,2017-03-29 00:12:00,6.50,83.0,61.0,7.4,-3.0,NaN,NaN
AL000009,2017-03-31 04:19:00,6.50,130.0,211.0,NaN,NaN,NaN,NaN
AL000300,2018-07-20 05:37:00,6.50,130.2,53.0,NaN,NaN,NaN,NaN
AL000376,2018-11-01 17:40:00,6.50,130.0,100.0,NaN,NaN,NaN,NaN
AL000528,2019-05-09 20:25:00,6.50,121.0,NaN,NaN,NaN,NaN,NaN
AL000328,2018-08-12 11:33:00,6.54,130.0,75.0,NaN,NaN,NaN,NaN
AL001027,2020-08-11 21:46:00,6.58,130.0,38.0,NaN,NaN,NaN,NaN
AL000289,2018-07-01 04:27:00,6.59,97.0,75.0,9.0,-28.0,0.4,NaN


In [29]:
# These are all correct
gas_frames_all[gas_frames_all['pH'] > 7.5].sort_values('pH', ascending=False)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AL000043,2017-05-02 21:12:00,7.600,20.0,67.0,19.6,-2.0,NaN,NaN
AL000357,2018-09-03 17:00:00,7.600,38.8,149.0,38.3,14.3,1.00,Capillaris
AL000526,2019-05-08 10:23:00,7.580,32.6,48.0,31.1,9.0,0.21,Capillaris
AL000357,2018-09-03 17:57:00,7.550,51.0,38.3,42.6,18.5,0.21,Capillaris
AL001086,2020-09-21 11:30:00,7.550,43.0,56.0,37.0,15.2,0.21,Capillaris
AL000689,2019-09-19 10:19:00,7.548,44.5,41.0,38.8,16.0,0.65,NaN
AL000114,2017-08-03 10:39:00,7.526,42.0,35.5,29.4,7.0,0.21,Capillaris
AL000526,2019-05-08 11:51:00,7.520,39.5,60.2,31.4,8.3,0.21,Capillaris
AL000918,2020-05-24 16:57:00,7.520,29.7,40.8,23.8,2.6,0.21,Capillaris


In [30]:
gas_frames_all.dropna(how='any')[gas_frames_all.dropna(how='any')['pCO2'] > 130].sort_values('pCO2', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,


In [31]:
# These are all correct
gas_frames_all[gas_frames_all['pCO2'] < 20].sort_values('pCO2', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AL000951,2020-06-19 04:12:00,7.380,15.0,83.0,15.1,-16.2,0.21,Capillaris
AL000729,2019-12-20 23:42:00,7.062,17.2,74.0,4.9,-25.0,NaN,NaN
AL000162,2017-11-25 10:41:00,7.357,19.0,53.7,15.3,-14.2,0.21,Capillaris


In [32]:
# These are all correct
gas_frames_all[gas_frames_all['ABE'] < -30].sort_values('ABE', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AL000528,2019-05-09 22:53:00,6.42,99.0,51.8,3.4,-34.0,0.7,Capillaris


In [33]:
# These are all correct
gas_frames_all[gas_frames_all['ABE'] > 20].sort_values('ABE', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AL000766,2020-01-31 11:23:00,7.430,68.0,46.0,45.1,20.8,0.30,Capillaris
AL000719,2019-12-15 21:53:00,7.377,79.4,47.0,46.7,22.0,0.60,NaN
AL000610,2019-07-05 21:36:00,7.270,50.5,23.0,-3.9,80.0,0.25,Capillaris


### 6. Export bood gases as Excel files

In [38]:
# Save blood gases into a multi-sheet Excel file
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'blood_gases_1_1100.xlsx'))
for case in sorted(gas_frames.keys()):
    gas_frames[case].to_excel(writer, sheet_name=case)
writer.close()

### 7. Export processed data as pickle files

In [40]:
with open(os.path.join(DATA_DUMP, 'blood_gases_1_1100.pickle'), 'wb') as handle:
    pickle.dump(gas_frames, handle, protocol=pickle.HIGHEST_PROTOCOL)